# ACDC Eval Metrics Comparison

This notebook aggregates evaluation metrics from all notebooks in `notebooks/eval/acdc/**` and builds comparison tables across `e0`, `e1`, and `e2`.

It parses saved notebook outputs, so run eval notebooks first to refresh numbers.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

COND_ORDER = ["fog", "night", "rain", "snow"]


def collect_output_text(nb):
    parts = []
    for cell in nb.get("cells", []):
        for out in cell.get("outputs", []) if isinstance(cell.get("outputs", []), list) else []:
            if "text" in out:
                t = out["text"]
                parts.append("".join(t) if isinstance(t, list) else str(t))
            data = out.get("data", {}) if isinstance(out, dict) else {}
            if isinstance(data, dict):
                for key in ("text/plain", "text/markdown", "text/html"):
                    if key in data:
                        v = data[key]
                        parts.append("".join(v) if isinstance(v, list) else str(v))
    return "\n".join(parts)


def extract_first_float(pattern, text, flags=0):
    m = re.search(pattern, text, flags)
    return float(m.group(1)) if m else np.nan


def extract_condition_miou_map(text):
    # Limit parsing to the per-condition metric section to avoid sample-count matches.
    section = re.search(
        r"Per-condition mIoU([\s\S]*?)(?:Class-wise IoU|GT-based IoU|Reference proxy metrics|$)",
        text,
        flags=re.IGNORECASE,
    )
    search_text = section.group(1) if section else text

    result = {}
    for cond in COND_ORDER:
        val = np.nan

        # Pandas plain text dataframe row format: "0   fog   0.6854"
        m_plain = re.search(
            rf"(?:^|\n)\s*\d+\s+{cond}\s+([0-9]*\.?[0-9]+)",
            search_text,
            flags=re.IGNORECASE,
        )
        if m_plain:
            val = float(m_plain.group(1))
        else:
            # HTML dataframe format: <td>fog</td><td>0.6854</td>
            m_html = re.search(
                rf"<td>{cond}</td>\s*<td>([0-9]*\.?[0-9]+)</td>",
                search_text,
                flags=re.IGNORECASE,
            )
            if m_html:
                val = float(m_html.group(1))

        result[cond] = val

    return result


def infer_model_name(task_name, file_stem):
    if "DinoV2" in task_name:
        return "DinoV2"
    if "SegFormer_B2" in task_name or "Segformer" in file_stem:
        return "SegFormer-B2"
    if "InternImage" in task_name:
        return "InternImage"
    if "DeepLabV3" in task_name:
        return "DeepLabV3"
    return file_stem.split("_")[0]


In [2]:
root = Path.cwd()
if (root / "acdc").exists():
    eval_root = root / "acdc"
elif (root / "notebooks" / "eval" / "acdc").exists():
    eval_root = root / "notebooks" / "eval" / "acdc"
else:
    raise FileNotFoundError("Could not locate notebooks/eval/acdc")

stage_order = ["e0", "e1", "e2"]

rows = []
for nb_path in sorted(eval_root.glob("**/*.ipynb")):
    nb = json.loads(nb_path.read_text())

    source_text = "\n".join(
        "".join(c.get("source", [])) if isinstance(c.get("source"), list) else str(c.get("source", ""))
        for c in nb.get("cells", [])
    )
    output_text = collect_output_text(nb)

    task = re.search(r'"task_name"\s*:\s*"([^"]+)"', source_text)
    weights = re.search(r'"weights_rel_path"\s*:\s*"([^"]+)"', source_text)
    task_name = task.group(1) if task else ""

    row = {
        "file": str(nb_path.relative_to(eval_root)),
        "stage": nb_path.parent.name,
        "model": infer_model_name(task_name, nb_path.stem),
        "task_name": task_name,
        "weights_rel_path": weights.group(1) if weights else "",
        "overall_miou": extract_first_float(r"mIoU\s+([0-9]*\.?[0-9]+)", output_text),
        "pixel_accuracy": extract_first_float(r"Pixel Accuracy\s+([0-9]*\.?[0-9]+)", output_text),
    }

    cond_map = extract_condition_miou_map(output_text)
    for cond in COND_ORDER:
        row[f"miou_{cond}"] = cond_map.get(cond, np.nan)

    rows.append(row)

metrics_df = pd.DataFrame(rows)
metrics_df["stage"] = pd.Categorical(metrics_df["stage"], categories=stage_order, ordered=True)
metrics_df = metrics_df.sort_values(["model", "stage", "file"]).reset_index(drop=True)
metrics_df


,file,stage,model,task_name,weights_rel_path,overall_miou,pixel_accuracy,miou_fog,miou_night,miou_rain,miou_snow
0,e0/DeepLabV3_ACDC_Evaluation.ipynb,e0,DeepLabV3,DeepLabV3Plus_Eval_ACDC_Val,weights/deeplab-cityscapes-epoch=47-val_miou=0...,0.301473,0.720400,0.442170,0.136303,0.329316,0.321498
1,e0/DinoV2_ACDC_Evaluation.ipynb,e0,DinoV2,DinoV2_Small_Eval_ACDC_Val,weights/dinov2-small-cityscapes-epoch=48-val_m...,0.543588,0.857451,0.688203,0.334964,0.526262,0.581612
2,e1/DinoV2_E1_ACDC_Evaluation.ipynb,e1,DinoV2,DinoV2_Small_E1_Eval_ACDC_Val,weights/dinov2-small-E1-cityscapes-epoch=47-va...,0.562019,0.868956,0.694020,0.350639,0.553994,0.596316
3,e2/DinoV2_E2_ACDC_Evaluation.ipynb,e2,DinoV2,DinoV2_Small_E2_Eval_ACDC_Val,weights/dinov2-small-E2-cityscapes-epoch=48-va...,0.560874,0.864216,0.680970,0.350990,0.550731,0.597685
4,e0/InternImage_ACDC_Evaluation.ipynb,e0,InternImage,InternImage_Tiny_Eval_ACDC_Val,weights/internimage-tiny-cityscapes-epoch=39-v...,0.389877,0.773503,0.537972,0.213678,0.430759,0.399685
5,e0/Segformer_ACDC_Evaluation.ipynb,e0,SegFormer-B2,SegFormer_B2_Eval_ACDC_Val,weights/segformer-b2-cityscapes-epoch=43-val_m...,0.446281,0.768507,0.582002,0.246120,0.472170,0.455311
6,e1/SegformerB2_E1_ACDC_Evaluation.ipynb,e1,SegFormer-B2,SegFormer_B2_E1_Eval_ACDC_Val,weights/segformer-b2-E1-cityscapes-epoch=48-va...,0.502201,0.827274,0.632992,0.274023,0.523213,0.527209
7,e2/SegformerB2_E2_ACDC_Evaluation.ipynb,e2,SegFormer-B2,SegFormer_B2_E2_Eval_ACDC_Val,weights/segformer-b2-E2-cityscapes-epoch=45-va...,0.500093,0.824918,0.635666,0.288102,0.529294,0.515760


In [3]:
summary_cols = ["overall_miou", "pixel_accuracy", "miou_fog", "miou_night", "miou_rain", "miou_snow"]
available_stages = [stage for stage in stage_order if stage in set(metrics_df["stage"].astype(str))]

pivot = metrics_df.pivot_table(index="model", columns="stage", values=summary_cols, aggfunc="first")
ordered_cols = [
    (metric, stage)
    for metric in summary_cols
    for stage in available_stages
    if (metric, stage) in pivot.columns
]
pivot = pivot.reindex(columns=pd.MultiIndex.from_tuples(ordered_cols))
pivot


def build_delta_frame(pivot_df, left_stage, right_stage):
    delta_df = pd.DataFrame(index=pivot_df.index)
    for metric in summary_cols:
        if (metric, left_stage) in pivot_df.columns and (metric, right_stage) in pivot_df.columns:
            delta_df[f"{metric}_delta_{right_stage}_minus_{left_stage}"] = (
                pivot_df[(metric, right_stage)] - pivot_df[(metric, left_stage)]
            )
    return delta_df


for left_stage, right_stage in [("e0", "e1"), ("e1", "e2"), ("e0", "e2")]:
    if left_stage in available_stages and right_stage in available_stages:
        print(f"Delta ({right_stage} - {left_stage}):")
        display(build_delta_frame(pivot, left_stage, right_stage))

missing = metrics_df[metrics_df["overall_miou"].isna()][["file", "task_name", "weights_rel_path"]]
if len(missing) > 0:
    print("Notebooks without parsed overall_miou in saved outputs:")
    display(missing)


Delta (e1 - e0):


/tmp/ipykernel_72024/2946353852.py:4: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = metrics_df.pivot_table(index="model", columns="stage", values=summary_cols, aggfunc="first")


,overall_miou_delta_e1_minus_e0,pixel_accuracy_delta_e1_minus_e0,miou_fog_delta_e1_minus_e0,miou_night_delta_e1_minus_e0,miou_rain_delta_e1_minus_e0,miou_snow_delta_e1_minus_e0
model,,,,,,
DeepLabV3,NaN,NaN,NaN,NaN,NaN,NaN
DinoV2,0.018431,0.011505,0.005817,0.015675,0.027732,0.014704
InternImage,NaN,NaN,NaN,NaN,NaN,NaN
SegFormer-B2,0.055920,0.058767,0.050990,0.027903,0.051043,0.071898


Delta (e2 - e1):


,overall_miou_delta_e2_minus_e1,pixel_accuracy_delta_e2_minus_e1,miou_fog_delta_e2_minus_e1,miou_night_delta_e2_minus_e1,miou_rain_delta_e2_minus_e1,miou_snow_delta_e2_minus_e1
model,,,,,,
DeepLabV3,NaN,NaN,NaN,NaN,NaN,NaN
DinoV2,-0.001145,-0.004740,-0.013050,0.000351,-0.003263,0.001369
InternImage,NaN,NaN,NaN,NaN,NaN,NaN
SegFormer-B2,-0.002108,-0.002356,0.002674,0.014079,0.006081,-0.011449


Delta (e2 - e0):


,overall_miou_delta_e2_minus_e0,pixel_accuracy_delta_e2_minus_e0,miou_fog_delta_e2_minus_e0,miou_night_delta_e2_minus_e0,miou_rain_delta_e2_minus_e0,miou_snow_delta_e2_minus_e0
model,,,,,,
DeepLabV3,NaN,NaN,NaN,NaN,NaN,NaN
DinoV2,0.017286,0.006765,-0.007233,0.016026,0.024469,0.016073
InternImage,NaN,NaN,NaN,NaN,NaN,NaN
SegFormer-B2,0.053812,0.056411,0.053664,0.041982,0.057124,0.060449
